# Data Collection

Collects the two raw data sources required by `03_main.ipynb`:

| Section | Output file | Skip if exists? |
|---------|-------------|-----------------|
| **Part A – Reddit** | `data/reddit/<id>.csv` → `data/full_manual_wsb.csv` | ✅ Yes |
| **Part B – SPY** | `data/spy/<range>.csv` → `data/full_manual_spy.csv` | ✅ Yes |

> **Note:** Pre-collected data is already committed to `data/`.  
> You only need to run this notebook if you want to extend the date range
> or refresh the dataset.  **Reddit credentials** (`.env`) are only required for Part A.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # make src/ importable


---
## Part A — Reddit / WallStreetBets Comments

**Methodology**
1. `submission_id` values are taken from the r/wallstreetbets daily discussion threads.
2. Only comments posted during NYSE market hours (Mon–Fri, 09:30–16:00 ET) are kept.
3. Each day's raw CSV is saved to `data/reddit/`, then merged into `data/full_manual_wsb.csv`.

Requires `CLIENT_ID`, `CLIENT_SECRET`, and `DEV_NAME` in `.env`.
Register an app at https://www.reddit.com/prefs/apps (type: **script**).


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
# Add one entry per daily discussion thread you want to scrape.
# {id: str, url: str}  (url is informational only)
SUBMISSIONS = [
    {"id": "1pkq67i", "url": "https://www.reddit.com/r/wallstreetbets/comments/1pkq67i/daily_discussion_thread_for_december_12_2025/"},
    # Add more threads here as needed …
]

REDDIT_RAW_DIR  = "../data/reddit"       # per-day raw CSVs
REDDIT_OUT_FILE = "../data/full_manual_wsb.csv"  # merged output

SKIP_IF_EXISTS = True   # set False to force re-scrape even when output exists


In [ ]:
# %pip install praw python-dotenv   # uncomment if not yet installed

import os, glob, time
import pandas as pd
import praw
from dotenv import load_dotenv
from datetime import datetime
from prawcore.exceptions import TooManyRequests

load_dotenv(dotenv_path=os.path.join('..', '.env'))

CLIENT_ID     = os.environ.get('CLIENT_ID')
CLIENT_SECRET = os.environ.get('CLIENT_SECRET')
DEV_NAME      = os.environ.get('DEV_NAME')


In [ ]:
def _build_reddit_client() -> praw.Reddit:
    """Return an authenticated read-only Reddit client."""
    if not all([CLIENT_ID, CLIENT_SECRET, DEV_NAME]):
        raise EnvironmentError(
            "Reddit credentials missing. Copy .env.example → .env and fill in "
            "CLIENT_ID, CLIENT_SECRET, DEV_NAME."
        )
    return praw.Reddit(
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
        user_agent=f"RedditStream v1 (by /u/{DEV_NAME})",
    )


def scrape_submission(reddit: praw.Reddit, submission_id: str) -> pd.DataFrame:
    """Expand the full comment tree for *submission_id* and return a DataFrame.

    Retries automatically on rate-limit (HTTP 429).
    """
    print(f"  Expanding comment tree for {submission_id} …")
    submission = reddit.submission(id=submission_id)

    while True:
        try:
            submission.comments.replace_more(limit=None)
            break
        except TooManyRequests as exc:
            wait = float(getattr(exc.response, 'headers', {}).get('retry-after', 60))
            print(f"  Rate-limited – sleeping {wait:.0f}s …")
            time.sleep(wait)

    rows = []
    for comment in submission.comments.list():
        if not isinstance(comment, praw.models.Comment):
            continue
        rows.append({
            "id":               comment.id,
            "author":           "[deleted]" if not comment.author else comment.author.name,
            "score":            comment.score,
            "created_utc":      comment.created_utc,
            "created_datetime": datetime.fromtimestamp(comment.created_utc).strftime('%Y-%m-%d %H:%M:%S'),
            "body":             comment.body.replace('\n', ' '),
        })
    return pd.DataFrame(rows)


def filter_market_hours(df: pd.DataFrame, dt_col: str = 'created_datetime') -> pd.DataFrame:
    """Return only rows that fall within NYSE market hours (Mon–Fri 09:30–16:00)."""
    dt = pd.to_datetime(df[dt_col], format='mixed')
    is_weekday    = dt.dt.dayofweek.between(0, 4)
    is_after_open = (dt.dt.hour > 9) | ((dt.dt.hour == 9) & (dt.dt.minute >= 30))
    is_before_close = (dt.dt.hour < 16) | ((dt.dt.hour == 16) & (dt.dt.minute == 0))
    return df[is_weekday & is_after_open & is_before_close].copy()


def merge_reddit_csvs(raw_dir: str) -> pd.DataFrame:
    """Concatenate all per-day CSVs in *raw_dir* into one DataFrame."""
    files = sorted(glob.glob(os.path.join(raw_dir, '*.csv')))
    if not files:
        raise FileNotFoundError(f"No CSV files found in {raw_dir!r}")
    return pd.concat((pd.read_csv(f) for f in files), ignore_index=True)


In [ ]:
os.makedirs(REDDIT_RAW_DIR, exist_ok=True)

if SKIP_IF_EXISTS and os.path.exists(REDDIT_OUT_FILE):
    print(f"SKIP – {REDDIT_OUT_FILE!r} already exists. Set SKIP_IF_EXISTS=False to re-scrape.")
else:
    reddit = _build_reddit_client()

    for entry in SUBMISSIONS:
        sid  = entry['id']
        dest = os.path.join(REDDIT_RAW_DIR, f"reddit_comments_{sid}.csv")

        if SKIP_IF_EXISTS and os.path.exists(dest):
            print(f"  SKIP {sid} – raw file already exists.")
            continue

        print(f"Scraping submission {sid} …")
        df_raw = scrape_submission(reddit, sid)
        df_raw.to_csv(dest, index=False)
        print(f"  Saved {len(df_raw):,} comments → {dest}")

    # Merge + filter
    df_all  = merge_reddit_csvs(REDDIT_RAW_DIR)
    df_mkt  = filter_market_hours(df_all)
    df_out  = df_mkt.rename(columns={'created_datetime': 'datetime', 'body': 'text'})
    df_out  = df_out[['datetime', 'score', 'text']].reset_index(drop=True)

    df_out.to_csv(REDDIT_OUT_FILE, index=False)
    print(f"\nDone – {len(df_out):,} market-hours comments saved → {REDDIT_OUT_FILE}")


### *(Optional)* Comment activity by hour


In [ ]:
# Load whichever data is available (freshly scraped or pre-existing)
_wsb = pd.read_csv(REDDIT_OUT_FILE, parse_dates=['datetime'])

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.hist(_wsb['datetime'].dt.hour, bins=24, range=(0, 24), edgecolor='black', alpha=0.7)
plt.title('WSB Comment Activity by Hour')
plt.xlabel('Hour of Day (24 h)')
plt.ylabel('Comment count')
plt.xticks(range(24))
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

print(f"Total market-hours comments: {len(_wsb):,}")


---
## Part B — SPY Intraday Price Data

Downloads 1-minute OHLCV bars for **$SPY** via `yfinance`, filters to market hours,
saves per-week CSVs to `data/spy/`, then merges into `data/full_manual_spy.csv`.

The date range must overlap with the Reddit comment window.
No API credentials required.


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
# Each entry defines one download range saved as data/spy/<filename>.csv
SPY_RANGES = [
    {"start": "2025-11-24", "end": "2025-11-28", "filename": "1124_1128.csv"},
    {"start": "2025-12-01", "end": "2025-12-05", "filename": "1201_1205.csv"},
    {"start": "2025-12-08", "end": "2025-12-12", "filename": "1208_1212.csv"},
    {"start": "2025-12-15", "end": "2025-12-19", "filename": "1215_1219.csv"},
    # Add more date windows here …
]

TICKER       = "SPY"
INTERVAL     = "1m"
MARKET_TZ    = "America/New_York"
SPY_RAW_DIR  = "../data/spy"
SPY_OUT_FILE = "../data/full_manual_spy.csv"

SKIP_IF_EXISTS = True   # set False to force re-download


In [ ]:
# %pip install yfinance   # uncomment if not yet installed

import glob, os
import pandas as pd
import yfinance as yf


In [ ]:
def fetch_spy(ticker: str, start: str, end: str,
              interval: str = '1m', tz: str = 'America/New_York') -> pd.DataFrame:
    """Download *ticker* OHLCV data and filter to NYSE market hours.

    Parameters
    ----------
    ticker   : Yahoo Finance ticker symbol (e.g. 'SPY')
    start    : inclusive start date, 'YYYY-MM-DD'
    end      : exclusive end date,   'YYYY-MM-DD'
    interval : bar size (default '1m')
    tz       : output timezone (default 'America/New_York')

    Returns
    -------
    pd.DataFrame with lowercase OHLCV columns, DatetimeIndex named 'timestamp'.
    """
    data = yf.download(tickers=ticker, start=start, end=end,
                       interval=interval, progress=False, auto_adjust=False)
    if data.empty:
        raise ValueError(f"No data returned for {ticker} {start} → {end}")

    # Flatten MultiIndex columns (yfinance ≥ 0.2)
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.droplevel(1)

    # Normalise timezone
    if data.index.tz is not None:
        data.index = data.index.tz_convert(tz)
    else:
        data.index = data.index.tz_localize(tz, ambiguous='NaT', nonexistent='NaT')

    # Market-hours filter
    h, m = data.index.hour, data.index.minute
    mask = ((h > 9) | ((h == 9) & (m >= 30))) & ((h < 16) | ((h == 16) & (m == 0)))
    data = data[mask].copy()

    # Clean up columns
    data = data.drop(columns=['Adj Close'], errors='ignore')
    data.columns = data.columns.str.lower()
    data.index.name = 'timestamp'
    return data


def merge_spy_csvs(raw_dir: str) -> pd.DataFrame:
    """Concatenate all per-week CSVs in *raw_dir* into one DataFrame."""
    files = sorted(glob.glob(os.path.join(raw_dir, '*.csv')))
    if not files:
        raise FileNotFoundError(f"No CSV files found in {raw_dir!r}")
    return pd.concat((pd.read_csv(f) for f in files), ignore_index=False)


In [ ]:
os.makedirs(SPY_RAW_DIR, exist_ok=True)

if SKIP_IF_EXISTS and os.path.exists(SPY_OUT_FILE):
    print(f"SKIP – {SPY_OUT_FILE!r} already exists. Set SKIP_IF_EXISTS=False to re-download.")
else:
    for rng in SPY_RANGES:
        dest = os.path.join(SPY_RAW_DIR, rng['filename'])

        if SKIP_IF_EXISTS and os.path.exists(dest):
            print(f"  SKIP {rng['filename']} – file already exists.")
            continue

        print(f"Fetching {TICKER} {rng['start']} → {rng['end']} …")
        df_week = fetch_spy(TICKER, rng['start'], rng['end'], INTERVAL, MARKET_TZ)
        df_week.to_csv(dest, index=True)
        print(f"  Saved {len(df_week):,} rows → {dest}")

    df_spy = merge_spy_csvs(SPY_RAW_DIR)
    df_spy.to_csv(SPY_OUT_FILE, index=False)
    print(f"\nDone – {len(df_spy):,} bars saved → {SPY_OUT_FILE}")


### *(Optional)* Sanity-check the merged SPY file


In [ ]:
_spy = pd.read_csv(SPY_OUT_FILE)
print(_spy.shape)
_spy.head()
